In [1]:
import apache_beam as beam
import os
import json
import importlib
import unittest
import numpy as np

from datetime import datetime
from apache_beam.transforms.util import WaitOn
from apache_beam.options.pipeline_options import PipelineOptions
from pyspark.sql.functions import date_format, col
from pyspark.sql.types import TimestampType

Failed to import GCSFileSystem; loading of this filesystem will be skipped. Error details: cannot import name 'storage' from 'google.cloud' (unknown location)


In [18]:
fromNotebook = True
configFile = "load_wwi_2014-01-01.json"
newCutoffDate = "2014-01-01"
tables = []

In [3]:
prefix = ""
if fromNotebook == False:
    prefix = "notebooks/"

file_path = os.path.join(os.getcwd(), f"{prefix}modules", 'sql_utilities.py')
spec = importlib.util.spec_from_file_location("sql_utils", file_path)
sql_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sql_utils)

In [21]:
f = open(configFile,)
config = json.load(f)
f.close()

source_db = config["source"]
destination_db = config["destination"]
initialCutoffDate = config["initialCutoffDate"]
spark_master = config["sparkMaster"]
spark_jars = config["sparkJars"]
spark_executor_memory = config["sparkExecutorMemory"]
spark_driver_memory = config["sparkDriverMemory"]
spark_executor_cores = config["sparkExecutorCores"]
spark_cores_max = config["sparkCoresMax"]
spark_network_timeout = config["sparkNetworkTimeout"]
spark_executor_heartbeat_interval = config["sparkExecutorHeartBeatInterval"]
if fromNotebook == True:
    tables = config["tables"]
    newCutoffDate = config["cutoffDate"]
load_tables = [
    {
        "name": table["name"],
        "schema": table["destination"]["schema"],
        "table": table["destination"]["table"]
    }
    for table in tables
]
runner_options = config["runnerOptions"][config["runner"]]
gcpCredential = config["gcpCredential"]

print(f"tables: {tables}")
print(f"source_db: {source_db}")
print(f"destination_db: {destination_db}") 
print(f"initialCutoffDate: {initialCutoffDate}")
print(f"spark_master: {spark_master}")
print(f"spark_jars: {spark_jars}")
print(f"spark_executor_memory: {spark_executor_memory}")
print(f"spark_driver_memory: {spark_driver_memory}")
print(f"spark_executor_cores: {spark_executor_cores}")
print(f"spark_cores_max: {spark_cores_max}")
print(f"spark_network_timeout: {spark_network_timeout}")
print(f"spark_executor_heartbeat_interval: {spark_executor_heartbeat_interval}")
print(f"newCutoffDate: {newCutoffDate}")
print(f"load_tables: {load_tables}")
print(f"runner_options: {runner_options}")
print(f"gcp_credential: {gcpCredential}")

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = gcpCredential


tables: [{'name': 'ApplicationCities', 'source': {'schema': 'Application', 'table': 'Cities', 'loadTable': 'scripts/load_wwi_mssql/load_application_cities.sql'}, 'destination': {'schema': 'dbo', 'table': 'Application_Cities', 'stagingTable': 'Application_Cities_Staging', 'createTable': 'scripts/load_wwi_mssql/create_application_cities.sql', 'modifyTable': [{'name': 'Add NewColumn column', 'tableUpdate': 'scripts/load_wwi_mssql/modify_cities1.sql', 'tableCreateStaging': 'scripts/load_wwi_mssql/modify_cities_staging1.sql', 'tableDataLoad': 'scripts/load_wwi_mssql/modify_cities_load1.sql', 'tableDataUpdate': 'scripts/load_wwi_mssql/modify_cities_update1.sql', 'tableDataTest': 'scripts/load_wwi_mssql/tests/modify_cities_load1.sql', 'tableCountDataTest': 'scripts/load_wwi_mssql/tests/validrange_modify_record_count.sql'}], 'insertTable': 'scripts/load_wwi_mssql/insert_record.sql', 'dropTable': 'scripts/load_wwi_mssql/drop_table.sql', 'deleteByLoadDateTable': 'scripts/load_wwi_mssql/delete_re

In [22]:
if source_db["databaseType"].startswith("spark-") and destination_db["databaseType"].startswith("spark-"):
    sql_utils.initialize_spark_session(
        master=spark_master,
        jdbc_url=source_db["conn"], 
        jars=spark_jars, 
        executor_memory=spark_executor_memory, 
        driver_memory=spark_driver_memory, 
        executor_cores=spark_executor_cores, 
        cores_max=spark_cores_max, 
        network_timeout=spark_network_timeout, 
        executor_heartbeat_interval=spark_executor_heartbeat_interval
    )

In [23]:
def get_last_cutoff_date(table, last_cutoff_dates):
    last_cutoff_date = next((item for item in last_cutoff_dates if item.get("TableName") == table), None)

    if last_cutoff_date is None:
        return initialCutoffDate
    else:
        return last_cutoff_date["LastCutoffDate"].strftime("%Y-%m-%d %H:%M:%S")

In [24]:
class LoadData(beam.DoFn):
    def __init__(self, database_type, conn_str, load_path, values_to_replace, tables_to_replace, show_sql = False, result_type = "tuple", set_timestamp_tostring = False, spark_load_sql = False, database = ""):
        self.database_type = database_type
        self.conn_str = conn_str
        self.load_path = load_path
        self.values_to_replace = values_to_replace
        self.tables_to_replace = tables_to_replace
        self.show_sql = show_sql
        self.result_type = result_type
        self.set_timestamp_tostring = set_timestamp_tostring
        self.spark_load_sql = spark_load_sql
        self.database = database

    def process(self, element):
        sql = sql_utils.get_sql_from_script(
            self.load_path, 
            self.values_to_replace, 
            self.tables_to_replace
        )

        if self.show_sql == True:
            print(f"LoadData sql: {sql}")
        
        return sql_utils.select_sql(
            self.conn_str, 
            sql, 
            self.database_type, 
            self.result_type, 
            self.database, 
            spark_jars, 
            spark_master, 
            self.set_timestamp_tostring, 
            False, 
            self.spark_load_sql
        )

In [25]:
def upsert_data_base(conn, database_type, database, upsert_path, values_to_replace, tables_to_replace, show_sql = False, spark_insert_sql = False):
    sql = sql_utils.get_sql_from_script(
        upsert_path, 
        values_to_replace, 
        tables_to_replace
    )
        
    if show_sql == True:
        print(f"""Upsert Data sql:
                {sql} 
                """)
    
    if database_type.startswith("spark-") == False:
        spark_insert_sql = False

    sql_utils.exec_sql(conn, sql, None, database_type, database, spark_master, spark_jars, spark_insert_sql)

In [26]:
class UpsertData(beam.DoFn):
    def __init__(self, database_type, conn_str, upsert_path, values_to_replace, tables_to_replace, yield_record = None, show_sql = False, yield_element = False, database = "", register_spark_table = None, spark_insert_sql = False):
        self.database_type = database_type
        self.conn_str = conn_str
        self.upsert_path = upsert_path
        self.values_to_replace = values_to_replace
        self.tables_to_replace = tables_to_replace
        self.yield_record = yield_record
        self.show_sql = show_sql
        self.yield_element = yield_element
        self.database = database
        self.register_spark_table = register_spark_table
        self.spark_insert_sql = spark_insert_sql

    def process(self, element):
        upsert_data_base(self.conn_str, self.database_type, self.database, self.upsert_path, self.values_to_replace, self.tables_to_replace, self.show_sql, self.spark_insert_sql)

        if self.register_spark_table is not None and self.database_type.startswith("spark-") == True:
            sql_utils.register_spark_table( 
                spark_master, 
                spark_jars, 
                self.register_spark_table["table_id"], 
                self.register_spark_table["view_name"],
                self.database_type,
                self.conn_str
            )

        if self.yield_record is not None:
            yield self.yield_record
        else:
            if self.yield_element == True:
                yield element

In [27]:
class CreateTables(beam.DoFn):
    def __init__(self, database_type, conn_str, sql, database, yield_record = None, show_sql = False):
        self.database_type = database_type
        self.conn_str = conn_str
        self.sql = sql
        self.database = database
        self.yield_record = yield_record
        self.show_sql = show_sql

    def process(self, element):
        if len(self.sql) > 0:
            if self.show_sql == True:
                print(f"""Upsert Data sql:
                    {self.sql} 
                    """)
                
                print(f"element: {element}")
                print(f"len element: {len(element)}")

            # print(f"self.database: {self.database}")
            sql_utils.exec_sql(self.conn_str, self.sql, None, self.database_type, self.database, spark_master, spark_jars)

        if self.yield_record is not None:
            yield self.yield_record

In [28]:
tc = unittest.TestCase()

class LoadAndWriteData(beam.DoFn):
    def __init__(self, load_config, write_config, assert_config, history_config, modify_config = None, create_staging_config = None, modify_update_config = None, drop_staging_config = None, yield_record = None):
        self.load_config = load_config
        self.write_config = write_config
        self.assert_config = assert_config
        self.history_config = history_config
        self.modify_config = modify_config
        self.create_staging_config = create_staging_config
        self.modify_update_config = modify_update_config
        self.drop_staging_config = drop_staging_config
        self.yield_record = yield_record

    def start_bundle(self):
        self.batch = []
        self.columns = []

    def load_data(self):
        load_sql = sql_utils.get_sql_from_script(
            self.load_config["path"], 
            self.load_config["values_to_replace"], 
            self.load_config["tables_to_replace"]
        )

        if self.load_config["show_sql"] == True:
            print(f"LoadAndWriteData load data sql: {load_sql}")

        result_type = "tuple"
        if self.load_config["database_type"].startswith("spark-"):
            result_type = "spark-dataframe"

        return sql_utils.select_sql(
            self.load_config["conn"], 
            load_sql, 
            self.load_config["database_type"], 
            result_type, 
            "", 
            spark_jars, 
            spark_master, 
            self.load_config["set_timestamp_tostring"], 
            True
        )
    
    def write_data(self, data):
        sql = self.get_sql(data)

        if self.write_config["show_sql"] == True:
            print(f"write_data sql: {sql}")

        if self.write_config["database_type"].startswith("spark-"):
            sql_utils.exec_sql_batch(
                self.write_config["conn"], 
                sql, 
                data, 
                self.write_config["database_type"],
                self.write_config["database"],
                self.write_config["table_id"])
        else:
            sql_utils.exec_sql_batch(
                self.write_config["conn"], 
                sql, 
                data[1:], 
                self.write_config["database_type"], 
                self.write_config["database"], 
                self.write_config["table_id"])
            
    def get_sql(self, data):
        sql = ""

        if self.write_config["database_type"] == "mssql":
            if len(data) > 0:
                columns = []
                
                for index, col_name in enumerate(data[0]):
                    columns.append(col_name)

                for excluded_column in self.write_config["excluded_columns"]:
                    columns.remove(excluded_column)

                values = []

                for column in columns:
                    excluded_column = next((item for item in self.write_config["specific_column_types"] if item['name'] == column), None)           

                    if excluded_column is None:
                        values.append("?")
                    else:
                        values.append(excluded_column["typeCast"])

                sql = sql_utils.get_sql_from_script(
                    self.write_config["path"], 
                    self.write_config["values_to_replace"], 
                    []
                )

                sql = sql_utils.replace_sql_values(
                    sql, 
                    [
                        { 
                            "name": "Columns",
                            "value": ",".join(columns)
                        },
                        {
                            "name": "ColumnValues",
                            "value": ",".join(values)
                        }
                    ]
                )

        return sql

    def assert_data(self, loaded_data):
        if config["test"] == True:
            testCount = config["testRecordCount"]
            count = 0
            data = []
            
            if self.assert_config["database_type"].startswith("spark-"):
                count = loaded_data.count()
                loaded_data = loaded_data.limit(testCount)
                
                if self.assert_config["set_timestamp_tostring"] == True:
                    timestamp_fields = [f.name for f in loaded_data.schema.fields if isinstance(f.dataType, TimestampType)]

                    for col_name in timestamp_fields:
                        # if col_name in loaded_data.columns:
                        loaded_data = loaded_data.withColumn(
                            col_name,
                            date_format(col(col_name), "yyyy-MM-dd HH:mm:ss.SSSSSS")
                        )

                loaded_data = loaded_data.toPandas()
                data = loaded_data.replace({np.nan: None}).to_dict(orient="records")
            else:
                count = len(loaded_data) - 1 
                loaded_data = loaded_data[:testCount + 1]
                cols = []
                
                for index, col_name in enumerate(loaded_data[0]):
                    cols.append({"index": index, "col_name": col_name})

                loaded_data.pop(0)

                data = []
                for item in loaded_data:
                    row = {}

                    for column in cols:
                        row[column["col_name"]] = item[column["index"]]
                    data.append(row)

            count_sql = sql_utils.get_sql_from_script(
                self.assert_config["count_path"],
                self.assert_config["values_to_replace"], 
                self.assert_config["tables_to_replace"]
            )

            expected_data_sql = sql_utils.get_sql_from_script(
                self.assert_config["load_path"], 
                self.assert_config["values_to_replace"], 
                self.assert_config["tables_to_replace"]
            )

            if self.assert_config["show_sql"] == True:
                print(f"count_sql: {count_sql}")
                print(f"expected_data_sql: {expected_data_sql}")

            expected_data_count = sql_utils.exec_sql_scalar(
                self.assert_config["conn"], 
                count_sql, 
                self.assert_config["database_type"], 
                self.assert_config["database"], 
                spark_master, 
                spark_jars
            )
            expected_data_sample = list(
                sql_utils.select_sql(
                    self.assert_config["conn"], 
                    expected_data_sql, 
                    self.assert_config["database_type"], 
                    "dictionary", 
                    self.assert_config["database"], 
                    spark_jars, 
                    spark_master, 
                    self.assert_config["set_timestamp_tostring"]
                )
            )
            
            tc.assertEqual(count, expected_data_count)
            tc.assertListEqual(data, expected_data_sample)
    
    def insert_load_history(self):
        upsert_data_base(
            self.history_config["conn"],
            self.history_config["database_type"],
            self.history_config["database"],
            self.history_config["upsert_path"],
            self.history_config["values_to_replace"],
            self.history_config["tables_to_replace"],
            self.history_config["show_sql"],
            self.history_config["spark_insert_sql"]
        )

    def modify_table(self):
        upsert_data_base(
            self.modify_config["conn"],
            self.modify_config["database_type"],
            self.modify_config["database"],
            self.modify_config["upsert_path"],
            self.modify_config["values_to_replace"],
            self.modify_config["tables_to_replace"],
            self.modify_config["show_sql"],
            self.modify_config["spark_insert_sql"]
        )

    def create_staging_table(self):
        upsert_data_base(
            self.create_staging_config["conn"],
            self.create_staging_config["database_type"],
            self.create_staging_config["database"],
            self.create_staging_config["upsert_path"],
            self.create_staging_config["values_to_replace"],
            self.create_staging_config["tables_to_replace"],
            self.create_staging_config["show_sql"],
            self.create_staging_config["spark_insert_sql"]
        )

    def modify_update_table(self):
        upsert_data_base(
            self.modify_update_config["conn"],
            self.modify_update_config["database_type"],
            self.modify_update_config["database"],
            self.modify_update_config["upsert_path"],
            self.modify_update_config["values_to_replace"],
            self.modify_update_config["tables_to_replace"],
            self.modify_update_config["show_sql"],
            self.modify_update_config["spark_insert_sql"]
        )

    def drop_staging_table(self):
        upsert_data_base(
            self.drop_staging_config["conn"],
            self.drop_staging_config["database_type"],
            self.drop_staging_config["database"],
            self.drop_staging_config["upsert_path"],
            self.drop_staging_config["values_to_replace"],
            self.drop_staging_config["tables_to_replace"],
            self.drop_staging_config["show_sql"],
            self.drop_staging_config["spark_insert_sql"]
        )

    def process(self, element):
        # Modify Table
        if self.modify_config is not None:
            self.modify_table()

        # Create Staging
        if self.create_staging_config is not None:
            self.create_staging_table() 

        # Load Data
        loaded_data = self.load_data()

        # Write Data
        self.write_data(loaded_data)

        if self.modify_update_config is not None:
            self.modify_update_table()
        
        # Assert Data
        self.assert_data(loaded_data)

        # Insert Load History
        self.insert_load_history()

        # Drop Staging
        if self.drop_staging_config is not None:
            self.drop_staging_table()             

        if self.yield_record is not None:
            yield self.yield_record

In [16]:
def get_last_cutoff_dates():
    sql = sql_utils.get_sql_from_script(
        config["loadHistory"]["loadLastCutoffDatesTable"], 
        [
            { "name": "Schema", "value": config["loadHistory"]["schema"] },
            { "name": "Table", "value": config["loadHistory"]["table"] },
            { "name": "Database", "value": destination_db["database"] }
        ], 
        []
    )

    return sql_utils.select_sql(destination_db["conn"], sql, destination_db["databaseType"], "dictionary", destination_db["database"], spark_jars, spark_master, False, False, True)

In [13]:
def get_create_tables_sql(tables, last_cutoff_dates, new_cutoff_date):
    create_table_sqls = []

    for table in tables:
        tableName = table["destination"]["table"]
        last_cutoff_date = get_last_cutoff_date(tableName, last_cutoff_dates)
        lastCutoffDateVal = datetime.strptime(last_cutoff_date, "%Y-%m-%d %H:%M:%S")
        newCutoffDateVal = datetime.strptime(new_cutoff_date, "%Y-%m-%d %H:%M:%S")

        if lastCutoffDateVal < newCutoffDateVal:
            create_table_sqls.append(
                sql_utils.get_sql_from_script(
                    table["destination"]["createTable"], 
                    [
                        { "name": "Database", "value": config["destination"]["database"] },
                        { "name": "Schema", "value": table["destination"]["schema"] },
                        { "name": "Table", "value": table["destination"]["table"] },
                        { "name": "NewCutoffDate", "value": new_cutoff_date },
                        { "name": "LHSchema", "value": config["loadHistory"]["schema"] },
                        { "name": "LHTable", "value": config["loadHistory"]["table"] },
                        { "name": "TableName", "value": table["destination"]["table"] }
                    ], 
                    []
                )
                
            )

    if len(create_table_sqls) > 0:
        return " ".join(create_table_sqls)
    
    return ""

In [30]:
last_cutoff_dates = get_last_cutoff_dates()
print(f"last_cutoff_dates: {last_cutoff_dates}")

options = PipelineOptions(runner_options)

with beam.Pipeline(options=options) as p:
    def modify_table(table, process_table, last_cutoff_dates, waiton_create_tables):
        tableName = table["destination"]["table"]
        last_cutoff_date = get_last_cutoff_date(tableName, last_cutoff_dates)
        
        modifyTableScripts = []
                    
        for modifyTableScript in table["destination"]["modifyTable"]:
            modifyTableScripts.append(
                sql_utils.get_sql_from_script(
                    config["modifyLoadHistory"]["loadUnprocessedModifyLoadHistoryScriptTable"],
                    [
                        { "name": "ScriptName", "value": modifyTableScript["tableUpdate"] },
                        { "name": "LoadScriptName", "value": modifyTableScript["tableDataLoad"] },
                        { "name": "UpdateScriptName", "value": modifyTableScript["tableDataUpdate"] },
                        { "name": "TestScriptName", "value": modifyTableScript["tableDataTest"] },
                        { "name": "TestCountScriptName", "value": modifyTableScript["tableCountDataTest"] },
                        { "name": "CreateStagingScriptName", "value": modifyTableScript["tableCreateStaging"] },
                        { "name": "Name", "value": modifyTableScript["name"] }
                    ]
                )
            )
        
        unprocessedScripts = sql_utils.select_spark_sql(
            destination_db["conn"],
            sql_utils.get_sql_from_script(
                config["modifyLoadHistory"]["loadUnprocessedModifyLoadHistoryTable"],
                [
                    { "name": "Database", "value": destination_db["database"] },
                    { "name": "Schema", "value": config["modifyLoadHistory"]["schema"] },
                    { "name": "Table", "value": config["modifyLoadHistory"]["table"] },
                    { "name": "TableName", "value": tableName },
                    { "name": "LoadDate", "value": last_cutoff_date },
                    { "name": "LHSchema", "value": config["loadHistory"]["schema"] },
                    { "name": "LHTable", "value": config["loadHistory"]["table"] },
                    { "name": "ModifyTableScripts", "value": " UNION ".join(modifyTableScripts) }
                ]
            ),
            destination_db["databaseType"],
            destination_db["database"],
            spark_master,
            spark_jars
        )
        
        for script in unprocessedScripts:
            print(f"Processing '{script["Name"]}' for table {tableName} for cutoff {newCutoffDate}.")

            process_table = (
                process_table
                | f"Modify Table {tableName} Load and Write Data" >>
                    beam.ParDo(
                        LoadAndWriteData(
                            modify_config={
                                "conn": destination_db["conn"],
                                "database_type": destination_db["databaseType"],
                                "database": destination_db["database"],
                                "upsert_path": script["ScriptName"],
                                "values_to_replace": [
                                    { "name": "Database", "value": destination_db["database"] },
                                    { "name": "Schema", "value": table["destination"]["schema"] },
                                    { "name": "Table", "value": table["destination"]["table"] },
                                ],
                                "tables_to_replace": [],
                                "spark_insert_sql": False,
                                "show_sql": False
                            },
                            create_staging_config={
                                "conn": destination_db["conn"],
                                "database_type": destination_db["databaseType"],
                                "database": destination_db["database"],
                                "upsert_path": script["CreateStagingScriptName"],
                                "values_to_replace": [
                                    { "name": "Database", "value": destination_db["database"] },
                                    { "name": "Schema", "value": table["destination"]["schema"] },
                                    { "name": "Table", "value": table["destination"]["stagingTable"] },
                                ],
                                "tables_to_replace": [],
                                "spark_insert_sql": False,
                                "show_sql": False
                            },
                            load_config={
                                "conn": source_db["conn"],
                                "database_type": source_db["databaseType"],
                                "path": script["LoadScriptName"],
                                "values_to_replace": [
                                    { "name": "Schema", "value": table["source"]["schema"] },
                                    { "name": "Table", "value": table["source"]["table"] },
                                    { "name": "LastCutoffDate", "value": last_cutoff_date }
                                ],
                                "tables_to_replace": [],
                                "set_timestamp_tostring": True,
                                "show_sql": False
                            },
                            write_config={
                                "conn": destination_db["conn"],
                                "database_type": destination_db["databaseType"],
                                "database": destination_db["database"],
                                "path": table["destination"]["insertTable"],
                                "table_id": f"{destination_db["database"]}.{table["destination"]["schema"]}.{table["destination"]["stagingTable"]}",
                                "values_to_replace": [
                                    { "name": "Database", "value": destination_db["database"] },
                                    { "name": "Schema", "value": table["destination"]["schema"] },
                                    { "name": "Table", "value": table["destination"]["stagingTable"] }
                                ],
                                "tables_to_replace": [],
                                "excluded_columns": [],
                                "specific_column_types": table["destination"]["specificColumnTypes"],
                                "show_sql": False 
                            },
                            modify_update_config={
                                "conn": destination_db["conn"],
                                "database_type": destination_db["databaseType"],
                                "database": destination_db["database"],
                                "upsert_path": script["UpdateScriptName"],
                                "values_to_replace": [
                                    { "name": "Database", "value": destination_db["database"] },
                                    { "name": "Schema", "value": table["destination"]["schema"] },
                                    { "name": "Table", "value": table["destination"]["table"] },
                                    { "name": "StagingTable", "value": table["destination"]["stagingTable"] },
                                    { "name": "LastCutoffDate", "value": last_cutoff_date }
                                ],
                                "tables_to_replace": [],
                                "spark_insert_sql": False,
                                "show_sql": False
                            },
                            assert_config={
                                "conn": destination_db["conn"],
                                "database_type": destination_db["databaseType"],
                                "database": destination_db["database"],
                                "count_path": script["TestCountScriptName"],
                                "load_path": script["TestScriptName"],
                                "values_to_replace": [
                                    { "name": "Database", "value": destination_db["database"] },
                                    { "name": "Schema", "value": table["destination"]["schema"] },
                                    { "name": "Table", "value": table["destination"]["table"] },
                                    { "name": "LastCutoffDate", "value": last_cutoff_date },
                                    { "name": "NewCutoffDate", "value": newCutoffDate },
                                    { "name": "NumberOfRows", "value": str(config["testRecordCount"]) }
                                ],
                                "tables_to_replace": [],
                                "set_timestamp_tostring": True,
                                "show_sql": False
                            },
                            history_config={
                                "conn": destination_db["conn"],
                                "database_type": destination_db["databaseType"],
                                "database": destination_db["database"],
                                "upsert_path": config["modifyLoadHistory"]["insertModifyLoadHistoryTable"],
                                "values_to_replace": [
                                    { "name": "Database", "value": destination_db["database"] },
                                    { "name": "Schema", "value": config["modifyLoadHistory"]["schema"] },
                                    { "name": "Table", "value": config["modifyLoadHistory"]["table"] },
                                    { "name": "TableName", "value": tableName },
                                    { "name": "LoadDate", "value": last_cutoff_date },
                                    { "name": "LastCutoffDate", "value": last_cutoff_date },
                                    { "name": "Status", "value": "Successful" },
                                    { "name": "Details", "value": "" },
                                    { "name": "ScriptName", "value": script["Name"] }
                                ],
                                "tables_to_replace": [],
                                "spark_insert_sql": False,
                                "show_sql": False
                            },
                            drop_staging_config={
                                "conn": destination_db["conn"],
                                "database_type": destination_db["databaseType"],
                                "database": destination_db["database"],
                                "upsert_path": table["destination"]["dropTable"],
                                "values_to_replace": [
                                    { "name": "Database", "value": destination_db["database"] },
                                    { "name": "Schema", "value": table["destination"]["schema"] },
                                    { "name": "Table", "value": table["destination"]["stagingTable"] },
                                ],
                                "tables_to_replace": [],
                                "spark_insert_sql": False,
                                "show_sql": False
                            },
                            yield_record={f"Successfully modified table {tableName} for {newCutoffDate}"}
                        )
                    )
                | f"Modify Table {tableName} WaitOn 1" >> WaitOn(waiton_create_tables)
            )

        return process_table

    # create Load History table
    create_load_history = (
        p 
        | "Input Create Load History Table" >> beam.Create([{ "PlaceHolder": "Temp 1 record" }])
        | "Create Load History Table" >> 
            beam.ParDo(
                UpsertData(
                    destination_db["databaseType"],
                    destination_db["conn"],
                    config["loadHistory"]["createLoadHistoryTable"],
                    [
                        { "name": "Database", "value": config["destination"]["database"] },
                        { "name": "Schema", "value": config["loadHistory"]["schema"] },
                        { "name": "Table", "value": config["loadHistory"]["table"] }
                    ],
                    [],
                    { "Successfully created Load History table" },
                    False,
                    False,
                    destination_db["database"] 
                )
            )
        | "Print Load History Table" >> beam.Map(print) 
    )

    # create Modify Load History table
    create_modify_load_history = (
        p 
        | "Input Create Modify Load History Table" >> beam.Create([{ "PlaceHolder": "Temp 1 record" }])
        | "Create Modify Load History Table" >> 
            beam.ParDo(
                UpsertData(
                    destination_db["databaseType"],
                    destination_db["conn"],
                    config["modifyLoadHistory"]["createModifyLoadHistoryTable"],
                    [
                        { "name": "Database", "value": config["destination"]["database"] },
                        { "name": "Schema", "value": config["modifyLoadHistory"]["schema"] },
                        { "name": "Table", "value": config["modifyLoadHistory"]["table"] }
                    ],
                    [],
                    { "Successfully created Modify Load History table" },
                    False,
                    False,
                    destination_db["database"]
                )
            )
        | "Print Modify Load History Table" >> beam.Map(print)
    )

    create_tables_sql = get_create_tables_sql(tables, last_cutoff_dates, newCutoffDate)
    create_tables = (
        p
        | "Input Create Tables" >> beam.Create([{ "PlaceHolder": "Temp 1 record" }])
        | "Create Tables" >> beam.ParDo(
            CreateTables(
                destination_db["databaseType"],
                destination_db["conn"],
                create_tables_sql,
                destination_db["database"],
                { f"Successfully created tables." },
                False
            )
        )
        | "Wait On History Tables" >> WaitOn(create_load_history, create_modify_load_history)
        | "Print Create Tables" >> beam.Map(print)
    )

    for table in tables:
        tableName = table["destination"]["table"]
        
        print(f"Processing table '{tableName}' for '{newCutoffDate}'.")

        last_cutoff_date = get_last_cutoff_date(tableName, last_cutoff_dates)
        lastCutoffDateVal = datetime.strptime(last_cutoff_date, "%Y-%m-%d %H:%M:%S")
        newCutoffDateVal = datetime.strptime(newCutoffDate, "%Y-%m-%d %H:%M:%S")
        
        if lastCutoffDateVal >= newCutoffDateVal:
            print(f"Table '{tableName}' has already been loaded for '{newCutoffDate}'.")
        else:

            process_table = p | f"Input Create Table for {tableName}" >> beam.Create([{ "PlaceHolder": "Temp 1 record" }])

            if len(table["destination"]["modifyTable"]) > 0:
                process_table = modify_table(table, process_table, last_cutoff_dates, create_tables)

            process_table = (
                process_table
                | f"Load and Write Data for Table {tableName}" >> 
                    beam.ParDo(
                        LoadAndWriteData(
                            load_config={
                                "conn": source_db["conn"],
                                "database_type": source_db["databaseType"],
                                "path": table["source"]["loadTable"],
                                "values_to_replace": [
                                    { "name": "Schema", "value": table["source"]["schema"] },
                                    { "name": "Table", "value": table["source"]["table"] },
                                    { "name": "NewCutoffDate", "value": newCutoffDate },
                                    { "name": "LastCutoffDate", "value": last_cutoff_date },
                                ],
                                "tables_to_replace": [],
                                "set_timestamp_tostring": True,
                                "show_sql": False
                            },
                            write_config={
                                "conn": destination_db["conn"],
                                "database_type": destination_db["databaseType"],
                                "database": destination_db["database"],
                                "path": table["destination"]["insertTable"],
                                "table_id": f"{destination_db["database"]}.{table["destination"]["schema"]}.{table["destination"]["table"]}",
                                "values_to_replace": [
                                    { "name": "Schema", "value": table["destination"]["schema"] },
                                    { "name": "Table", "value": table["destination"]["table"] },
                                    { "name": "NewCutoffDate", "value": newCutoffDate },
                                    { "name": "LastCutoffDate", "value": last_cutoff_date }
                                ],
                                "tables_to_replace": [],
                                "excluded_columns": [],
                                "specific_column_types": table["destination"]["specificColumnTypes"],
                                "show_sql": False 
                            },
                            assert_config={
                                "conn": destination_db["conn"],
                                "database_type": destination_db["databaseType"],
                                "database": destination_db["database"],
                                "count_path": table["destination"]["testCountTable"],
                                "load_path": table["destination"]["testLoadTable"],
                                "values_to_replace": [
                                    { "name": "Database", "value": destination_db["database"] },
                                    { "name": "Schema", "value": table["destination"]["schema"] },
                                    { "name": "Table", "value": table["destination"]["table"] },
                                    { "name": "NewCutoffDate", "value": newCutoffDate },
                                    { "name": "LastCutoffDate", "value": last_cutoff_date },
                                    { "name": "NumberOfRows", "value": str(config["testRecordCount"]) }
                                ],
                                "tables_to_replace": [],
                                "set_timestamp_tostring": True,
                                "show_sql": False
                            },
                            history_config={
                                "conn": destination_db["conn"],
                                "database_type": destination_db["databaseType"],
                                "database": destination_db["database"],
                                "upsert_path": config["loadHistory"]["insertLoadHistoryTable"],
                                "values_to_replace": [
                                    { "name": "TableName", "value": table["destination"]["table"] },
                                    { "name": "LoadDate", "value": newCutoffDate },
                                    { "name": "Status", "value": "Successful" },
                                    { "name": "Details", "value": "" },
                                    { "name": "Database", "value": destination_db["database"] },
                                    { "name": "Schema", "value": config["loadHistory"]["schema"] },
                                    { "name": "Table", "value": config["loadHistory"]["table"] }
                                ],
                                "tables_to_replace": [],
                                "spark_insert_sql": False,
                                "show_sql": False
                            }
                        )
                    )
                | f"WaitOn Table {tableName}" >> WaitOn(create_tables)
                | f"Result Table {tableName}" >> beam.Map(print)
            )

last_cutoff_dates: [{'TableName': 'Application_Cities', 'LastCutoffDate': datetime.datetime(2014, 1, 1, 0, 0)}, {'TableName': 'Application_Cities_Archive', 'LastCutoffDate': datetime.datetime(2014, 1, 1, 0, 0)}, {'TableName': 'Application_Countries', 'LastCutoffDate': datetime.datetime(2014, 1, 1, 0, 0)}, {'TableName': 'Application_Countries_Archive', 'LastCutoffDate': datetime.datetime(2014, 1, 1, 0, 0)}, {'TableName': 'Application_DeliveryMethods', 'LastCutoffDate': datetime.datetime(2014, 1, 1, 0, 0)}, {'TableName': 'Application_DeliveryMethods_Archive', 'LastCutoffDate': datetime.datetime(2014, 1, 1, 0, 0)}, {'TableName': 'Application_PaymentMethods', 'LastCutoffDate': datetime.datetime(2014, 1, 1, 0, 0)}, {'TableName': 'Application_PaymentMethods_Archive', 'LastCutoffDate': datetime.datetime(2014, 1, 1, 0, 0)}, {'TableName': 'Application_People', 'LastCutoffDate': datetime.datetime(2014, 1, 1, 0, 0)}, {'TableName': 'Application_People_Archive', 'LastCutoffDate': datetime.datetime(